# Exercise 03 — SOLUTION: Adaptive Attacks Against Defenses

## Background

See student notebook.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

model = torch.hub.load('chenyaofo/pytorch-cifar-models', 'cifar10_resnet20', pretrained=True)
model.eval()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])
])
testset = torchvision.datasets.CIFAR10('../data', train=False, download=True, transform=transform)
images, labels = next(iter(torch.utils.data.DataLoader(testset, batch_size=50, shuffle=False)))
targets = (labels + 3) % 10
IMG_MIN, IMG_MAX = images.min().item(), images.max().item()

# Store clean images for perturbation-based detection
clean_images = images.clone()

# ── Simple heuristic detector ──
# Measures high-frequency content of the PERTURBATION (x - clean_images).
# Clean images score 0; adversarial perturbations score > 0 if noisy.
def high_freq_power(x):
    """High-frequency power of the perturbation relative to the clean image."""
    delta = x - clean_images
    kernel = (torch.ones(1,1,3,3)/9).expand(3,-1,-1,-1)
    smoothed = nn.functional.conv2d(delta, kernel, padding=1, groups=3)
    return ((delta - smoothed)**2).mean(dim=[1,2,3])

def detect(x, threshold=0.001):
    """Returns 1 (adversarial) or 0 (clean) per image."""
    return (high_freq_power(x) > threshold).float()

def eval_attack(x_adv, true_labels, targets, label):
    acc = (model(x_adv).argmax(1) == true_labels).float().mean()
    tgt = (model(x_adv).argmax(1) == targets).float().mean()
    det = detect(x_adv).mean()
    print(f"{label:35s}  clean-acc={acc:.0%}  target-hit={tgt:.0%}  detection-rate={det:.0%}")

# Baseline standard targeted PGD
def pgd_targeted_std(images, targets, epsilon=0.05, alpha=0.005, steps=40):
    x = images.clone().detach()
    x = x + torch.empty_like(x).uniform_(-epsilon, epsilon)
    x = torch.clamp(x, IMG_MIN, IMG_MAX)
    for _ in range(steps):
        x = x.detach().requires_grad_(True)
        loss = nn.CrossEntropyLoss()(model(x), targets)
        loss.backward()
        with torch.no_grad():
            x = x - alpha * x.grad.sign()
            delta = torch.clamp(x - images, -epsilon, epsilon)
            x = torch.clamp(images + delta, IMG_MIN, IMG_MAX)
    return x.detach()

x_std = pgd_targeted_std(images, targets)
eval_attack(x_std, labels, targets, "Standard targeted PGD")
print()
print(f"Clean image power:  {high_freq_power(images).mean().item():.6f}   (always 0 — no perturbation)")
print(f"Standard PGD power: {high_freq_power(x_std).mean().item():.6f}   (noisy perturbation → detected)")
print()
print("Notice: standard PGD has a high detection rate because")
print("it creates high-frequency perturbation patterns.")

In [2]:
print(high_freq_power(images).mean().item())   # clean image power
print(high_freq_power(x_std).mean().item())    # standard PGD power

0.065131276845932
0.065990149974823


## Part A — Solution

In [3]:
def pgd_evasion_attack(images, targets, epsilon=0.05, alpha=0.005, steps=80, lam=500.0):
    # SOLUTION
    x = images.clone().detach()
    x = x + torch.empty_like(x).uniform_(-epsilon, epsilon)
    x = torch.clamp(x, IMG_MIN, IMG_MAX)
    for _ in range(steps):
        x = x.detach().requires_grad_(True)
        l_clf = nn.CrossEntropyLoss()(model(x), targets)
        l_det = high_freq_power(x).mean()
        loss = l_clf + lam * l_det
        loss.backward()
        with torch.no_grad():
            x = x - alpha * x.grad.sign()
            delta = torch.clamp(x - images, -epsilon, epsilon)
            x = torch.clamp(images + delta, IMG_MIN, IMG_MAX)
    return x.detach()

x_evasion = pgd_evasion_attack(images, targets)
eval_attack(x_evasion, labels, targets, "Evasion attack (joint loss)")


In [4]:
print(high_freq_power(images).mean().item())   # clean image power
print(high_freq_power(x_std).mean().item())    # standard PGD power


0.065131276845932
0.065990149974823


## Part B — Solution

In [ ]:
def random_preprocess(x):
    """Randomised preprocessing defense: strong brightness jitter + Gaussian noise."""
    factor = float(np.random.uniform(0.70, 1.30))   # ±30% brightness
    noise  = torch.randn_like(x) * 0.10              # σ=0.10 noise
    return torch.clamp(x * factor + noise, IMG_MIN, IMG_MAX)

def eval_random_defense(x_adv, targets, n_trials=30):
    """Evaluate adversarial success rate under random preprocessing (average over trials)."""
    hits = 0
    with torch.no_grad():
        for _ in range(n_trials):
            hits += (model(random_preprocess(x_adv)).argmax(1) == targets).float().sum().item()
    return hits / (n_trials * len(targets))

with torch.no_grad():
    hit_no_def = (model(x_std).argmax(1) == targets).float().mean().item()
print(f"Standard PGD — no defense:                    target-hit={hit_no_def:.0%}")
print(f"Standard PGD — under random defense:          target-hit={eval_random_defense(x_std, targets):.0%}")
print()

def pgd_eot_attack(images, targets, epsilon=0.05, alpha=0.005, steps=60, K=16):
    # SOLUTION
    x = images.clone().detach()
    x = x + torch.empty_like(x).uniform_(-epsilon, epsilon)
    x = torch.clamp(x, IMG_MIN, IMG_MAX)
    for _ in range(steps):
        x_leaf = x.detach().requires_grad_(True)
        acc_grad = torch.zeros_like(x)
        for _ in range(K):
            x_t = random_preprocess(x_leaf)
            loss = nn.CrossEntropyLoss()(model(x_t), targets)
            loss.backward()
            acc_grad += x_leaf.grad.detach()
            x_leaf.grad = None
        avg_grad = acc_grad / K
        with torch.no_grad():
            x = x - alpha * avg_grad.sign()
            delta = torch.clamp(x - images, -epsilon, epsilon)
            x = torch.clamp(images + delta, IMG_MIN, IMG_MAX)
    return x.detach()

x_eot = pgd_eot_attack(images, targets)
print(f"EoT PGD   — under random defense:             target-hit={eval_random_defense(x_eot, targets):.0%}")

## Discussion

In [6]:
print("""
=== Discussion ===

1. Why does standard PGD have a high detection rate?
   Standard PGD optimises only the classification loss; the resulting perturbations have
   high-frequency checkerboard patterns that are easily flagged by a noise detector.

2. Why does the joint-loss attack evade the detector?
   The added L_det term penalises high-frequency energy, so the optimiser is forced
   to spread perturbations more smoothly — sacrificing some attack power for stealthiness.

3. Why does EoT succeed against randomised preprocessing?
   Standard PGD optimises for ONE realisation of the preprocessing. Under a different
   realisation the perturbation lands in a different place and the attack fails.
   EoT averages gradients over K realisations, so the update direction is robust
   to the randomness — the perturbation 'works on average'.

4. What does this imply about defense papers?
   A defense should always be evaluated against an adaptive adversary that knows and
   optimises against the defense mechanism. Non-adaptive evaluation systematically
   overestimates robustness (Carlini et al., 2019).
""")



=== Discussion ===

1. Why does standard PGD have a high detection rate?
   Standard PGD optimises only the classification loss; the resulting perturbations have
   high-frequency checkerboard patterns that are easily flagged by a noise detector.

2. Why does the joint-loss attack evade the detector?
   The added L_det term penalises high-frequency energy, so the optimiser is forced
   to spread perturbations more smoothly — sacrificing some attack power for stealthiness.

3. Why does EoT succeed against randomised preprocessing?
   Standard PGD optimises for ONE realisation of the preprocessing. Under a different
   realisation the perturbation lands in a different place and the attack fails.
   EoT averages gradients over K realisations, so the update direction is robust
   to the randomness — the perturbation 'works on average'.

4. What does this imply about defense papers?
   A defense should always be evaluated against an adaptive adversary that knows and
   optimises against th